# FLORES-200 藏中对齐与 Unicode 检查

**目标**：验证本地 `bod_Tibt` 与 `zho_Hans` 的 dev/devtest 是否可以作为严格对齐的评测集。

成功条件：两侧行数与唯一 ID 数符合官方记录；ID、来源元数据和划分完全对齐；不存在空句；原始文件哈希稳定。重复与 Unicode 现象只记录，不在原始数据上自动修正。


In [ ]:
# Setup：只使用标准库；Parquet 检查由项目内轻量环境执行。
from pathlib import Path
import csv
import json
import subprocess

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'requirements-cpu.txt').is_file():
            return candidate
    raise FileNotFoundError('未找到 nlp-project 项目根目录')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
VALIDATOR = PROJECT_ROOT / 'src' / 'validate_flores.py'
SUMMARY_PATH = PROJECT_ROOT / 'reports' / 'flores200_local_full_check.json'
SAMPLE_PATH = PROJECT_ROOT / 'reports' / 'flores200_manual_check_20.csv'
assert VENV_PYTHON.is_file(), '请先按项目说明创建 .venv'
assert VALIDATOR.is_file()
{'project_root': str(PROJECT_ROOT), 'validator': str(VALIDATOR)}


## 实验计划

1. 以 256 MB 内存上限、2 个线程读取四个本地 Parquet 文件。
2. 核对行数、唯一 ID、缺失 ID、URL、领域、主题、图片标记和超链接标记。
3. 统计空句、精确重复、跨划分重叠和基础 Unicode 特征。
4. 等距抽取 dev 与 devtest 各 10 条，供人工评价对齐、忠实度、中文流畅度和风格。


In [ ]:
# Minimal baseline：重新生成检查结果，过程不会修改原始数据。
completed = subprocess.run(
    [str(VENV_PYTHON), str(VALIDATOR), '--memory-limit', '256MB', '--threads', '2'],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(completed.stdout.strip())


## 结果摘要

下面只显示关键统计，不在笔记本中打印大段藏文或中文原文。完整机器可读结果位于 `reports/flores200_local_full_check.json`。


In [ ]:
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
compact = {
    'critical_checks_passed': summary['passed_critical_alignment_checks'],
    'split_checks': summary['split_checks'],
    'exact_duplicates': summary['exact_duplicates'],
    'cross_split_overlap': summary['exact_cross_split_overlap'],
    'unicode': summary['unicode'],
    'domains': summary['domains'],
}
print(json.dumps(compact, ensure_ascii=False, indent=2))


In [ ]:
# 人工检查表结构验证；这里只展示索引和字符数，不提前影响人工判断。
with SAMPLE_PATH.open(encoding='utf-8-sig', newline='') as handle:
    manual_rows = list(csv.DictReader(handle))
sample_overview = [
    {
        'split': row['split'],
        'row_index': int(row['row_index']),
        'id': int(row['id']),
        'domain': row['domain'],
        'topic': row['topic'],
        'tibetan_chars': len(row['tibetan']),
        'chinese_chars': len(row['chinese']),
    }
    for row in manual_rows
]
print(f'人工检查条数：{len(manual_rows)}')
sample_overview[:3]


## 当前结论与下一步

- 2,009 组句子的关键对齐检查通过，dev 与 devtest 之间没有精确重复句对。
- 藏文侧有 4 个源句各出现两次，但对应不同中文译文；评测时保留，并在误差分析中单独标记。
- 10 个藏文句子会被 NFC 改写，主要涉及藏文组合/预组合字符。原始评测文本保持不变；模型输入是否 NFC 规范化必须作为明确实验变量。
- `topic` 标签存在大小写和拼写不一致，只能在分析副本中规范化，不能修改原始测试数据。
- 下一步由人工填写 `reports/flores200_manual_check_20.csv`，再决定其是否足以作为课程项目的通用域测试集。
